## This notebook is trying to register a low res DESI and high RES Xenium OMI TIFF images together

In [131]:
import warnings
warnings.filterwarnings("ignore")

import spatialdata as sd
from spatialdata_io import xenium
from pathlib import Path
from spatialdata.models import Image2DModel
from spatialdata.transformations import Scale
import tifffile
import napari
import numpy as np
import pandas as pd

In [107]:
user_home = Path.home()

# Define the specific project folder
project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "xenium"
msi_project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "msi"

raw_data_path = project_dir / "raw" / "output-XETG00169__0055588__55588_region_4__20250418__182706"
desi_img_data_path = msi_project_dir/ "raw"/ "3D_DESI_F5_Pos mode_40um_F5_5pos_40um_Features110425.ome.tif"

In [108]:
raw_data_path

PosixPath('/Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/raw/output-XETG00169__0055588__55588_region_4__20250418__182706')

In [109]:
sdata = xenium(raw_data_path)

INFO     reading                                                                                                   
         /Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/raw/output-XETG00169__0055588__5
         5588_region_4__20250418__182706/cell_feature_matrix.h5                                                    


In [110]:
sdata

SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 20495, 19973), (5, 10247, 9986), (5, 5123, 4993), (5, 2561, 2496), (5, 1280, 1248)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
│     └── 'nucleus_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (52665, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (52665, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (48131, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (52665, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), cell_circles (Shapes), nucleus_boundaries (Shapes)

In [111]:
desi_img = tifffile.imread(desi_img_data_path)

In [112]:
desi_img.shape

(88, 107, 102)

In [113]:
desi_img[87]

array([[ 59,  54,  22, ...,  12,  45,  74],
       [ 70, 180,   4, ...,   5,  86,  33],
       [ 82,  52, 142, ..., 437, 298,  20],
       ...,
       [ 11,  43, 140, ..., 245, 228, 475],
       [100,  32,  42, ..., 171,  92,  65],
       [105,  28,  75, ..., 208, 342, 356]],
      shape=(107, 102), dtype=uint16)

In [114]:
print("\n\n=== Finding DESI Channels ===")
print("We need to find channels for m/z: 616.25, 772.57, 893.75")
print("\nLet's look at some channels to identify patterns...")

# Quick visualization of several channels to find the right ones
viewer = napari.Viewer()

# Add a few DESI channels to explore
# Start with some spread across the range
test_channels = [3,17,23,25]

for i in test_channels:
    viewer.add_image(
        desi_img[i], 
        name=f"DESI_channel_{i}",
        visible=(i == 0),  # Only first one visible by default
        colormap='viridis'
    )

print(f"\nOpened napari with channels: {test_channels}")
print("Toggle through channels to find ones with clear tissue structure")
print("Note which channel numbers show the morphology you saw in QuPath")

napari.run()



=== Finding DESI Channels ===
We need to find channels for m/z: 616.25, 772.57, 893.75

Let's look at some channels to identify patterns...

Opened napari with channels: [3, 17, 23, 25]
Toggle through channels to find ones with clear tissue structure
Note which channel numbers show the morphology you saw in QuPath


In [115]:
channel_indices = [3,17,23,25]
desi_selected = desi_img[channel_indices, :, :] 

In [116]:
# Create SpatialData image with proper coordinate system
desi_spatial = Image2DModel.parse(
    desi_selected,
    dims=("c", "y", "x"),
    transformations={
        "global": Scale(
            [40.0, 40.0],  # 40 μm per pixel
            axes=("y", "x")
        )
    },
    c_coords=["mz_204.13", "heme_B_616.25", "PE_PS_731.65", "PE_PS_734.60"]
)

In [117]:
# Add to spatialdata object
sdata.images["desi"] = desi_spatial

In [118]:
print("\nDESI image added to SpatialData object!")
print(f"Available images now: {list(sdata.images.keys())}")


DESI image added to SpatialData object!
Available images now: ['morphology_focus', 'desi']


In [119]:
sdata.images['desi']

<xarray.DataArray 'image' (c: 4, y: 107, x: 102)> Size: 87kB
dask.array<array, shape=(4, 107, 102), dtype=uint16, chunksize=(4, 107, 102), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U13 208B 'mz_204.13' 'heme_B_616.25' ... 'PE_PS_734.60'
  * y        (y) float64 856B 0.5 1.5 2.5 3.5 4.5 ... 103.5 104.5 105.5 106.5
  * x        (x) float64 816B 0.5 1.5 2.5 3.5 4.5 ... 97.5 98.5 99.5 100.5 101.5
Attributes:
    transform:  {'global': Scale (y, x)\n    [40. 40.]}

### Below is the commented code to attempt and write the spatial data object in a .zarr format for faster computation

In [120]:
# # Workaround: spatialdata 0.4.0 bug — dask's to_parquet() tries to JSON-serialize
# # the .attrs on the points DataFrame, which contains non-serializable Scale/Affine
# # transformation objects. Monkey-patch write_points to strip attrs only around the
# # to_parquet() call, so _get_transformations() still works normally.
# import spatialdata._io.io_points as _io_points
# _original_write_points = _io_points.write_points

# def _patched_write_points(points, group, name, group_type="ngff:points", format=_io_points.CurrentPointsFormat()):
#     saved = points.attrs.copy()
#     original_to_parquet = points.to_parquet

#     def safe_to_parquet(*args, **kwargs):
#         points.attrs.clear()
#         points.attrs["spatialdata_attrs"] = saved.get("spatialdata_attrs", {})
#         try:
#             return original_to_parquet(*args, **kwargs)
#         finally:
#             points.attrs.clear()
#             points.attrs.update(saved)

#     points.to_parquet = safe_to_parquet
#     try:
#         _original_write_points(points, group, name, group_type, format)
#     finally:
#         points.to_parquet = original_to_parquet

# _io_points.write_points = _patched_write_points

# output_zarr_path = project_dir / "processed" / "desi_xenium_register.zarr"
# sdata.write(output_zarr_path, overwrite=True)

### Using Napari-Spatial Data to visualize

In [121]:
from napari_spatialdata import Interactive

In [122]:
# interactive = Interactive(sdata)
# interactive.run()

In [123]:
from spatialdata.models import get_channel_names

In [124]:
# 1. Create Xenium reference - combine vessel + membrane markers
morph = sdata.images['morphology_focus']

channel_names = get_channel_names(morph)
channel_names

[np.str_('DAPI'),
 np.str_('ATP1A1/CD45/E-Cadherin'),
 np.str_('18S'),
 np.str_('AlphaSMA/Vimentin'),
 np.str_('dummy')]

In [125]:
# The full resolution data is typically at 'scale0'
morph_full = morph['scale0'].to_dataset()

In [126]:
print(f"Spatial dimensions: y={morph_full.dims['y']}, x={morph_full.dims['x']}")

Spatial dimensions: y=20495, x=19973


In [127]:
# The actual array is accessed via the data variable (usually called something like 'image' or the dataset name)
# Let's find the data variable name:
print(f"\nData variables: {list(morph_full.data_vars)}")


Data variables: ['image']


In [128]:
morph_array = morph_full['image']

# Extract relevant channels in scale0 which is the highest resolution we have
# Adjust indices based on actual channel order in your file
alphasma_vim = morph_array.sel(c='AlphaSMA/Vimentin').values  # Vessel marker
atp1a1 = morph_array.sel(c='ATP1A1/CD45/E-Cadherin').values  # Membrane marker
dapi = morph_array.sel(c='DAPI').values  # Nuclei marker

In [129]:
## similar to above extract the DESI channels as well
desi = sdata.images['desi']
desi_flipped = np.flip(desi,axis=2)
mz_204 = desi_flipped.sel(c='mz_204.13').values
heme_b = desi_flipped.sel(c='heme_B_616.25').values
pe_ps_731 = desi_flipped.sel(c='PE_PS_731.65').values
pe_ps_734 = desi_flipped.sel(c='PE_PS_734.60').values

In [130]:
viewer = napari.Viewer()

viewer.add_image(
    atp1a1,
    name='ATP1A1 (membranes)',
    colormap='green',
    scale=[0.2125, 0.2125],
    blending='additive',
    visible=True
)

viewer.add_image(
    alphasma_vim,
    name='AlphaSMA/Vimentin (vessels)',
    colormap='magenta',
    scale=[0.2125, 0.2125],
    blending='additive',
    visible=True
)

viewer.add_image(
    dapi,
    name='DAPI (nuclei)',
    colormap='blue',
    scale=[0.2125, 0.2125],  # Xenium pixel size
    blending='additive',
    visible=True
)

# DESI channels at 40 μm/pixel
viewer.add_image(
    mz_204,
    name='DESI mz_204.13',
    colormap='cyan',
    scale=[40.0, 40.0],
    blending='additive',
    visible=False
)

viewer.add_image(
    heme_b,
    name='DESI Heme B (616.25)',
    colormap='yellow',
    scale=[40.0, 40.0],
    blending='additive',
    visible=False
)

viewer.add_image(
    pe_ps_731,
    name='DESI PE/PS (731.65)',
    colormap='red',
    scale=[40.0, 40.0],
    blending='additive',
    visible=False
)

viewer.add_image(
    pe_ps_734,
    name='DESI PE/PS (734.60)',
    colormap='bop orange',
    scale=[40.0, 40.0],
    blending='additive',
    visible=False
)

<Image layer 'DESI PE/PS (734.60)' at 0x4b2f3b730>